# Step 06 – Learning curves within the training partition

Paper 2: *Benchmarking Ten Machine-Learning Algorithms for Mediterranean Forest Mapping with Sentinel-2* (Remote Sensing, 2026).

- **In the paper:** Section 3.2 · Figure 6, Figure S4
- **Needs:** steps 03 and 04
- **Writes:** `RESULTS_DIR/learning_curves/`

Run the configuration cell below first.

In [ ]:
# ---------------------------------------------------------------- paths
# The data are not included in this repository. Set the two folders below to
# where the files are on your computer (or on your Google Drive in Colab).
#   DATA_DIR    : input data (training samples, Sentinel-2 stack, map tiles, rasters)
#   RESULTS_DIR : outputs written by the notebooks. Later steps read the outputs
#                 of earlier ones (e.g. RESULTS_DIR/train_test_gap/ from step 04).
import os
DATA_DIR    = 'data/'
RESULTS_DIR = 'results/'

# Google Colab: mount Drive and point the folders to it, for example
# from google.colab import drive; drive.mount('/content/drive')
# DATA_DIR    = '/content/drive/MyDrive/<your_folder>/'
# RESULTS_DIR = '/content/drive/MyDrive/<your_folder>/results/'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Files and folders used by this notebook:
#   DATA_DIR / S2_Features_For_Correlation_v3.csv
#   RESULTS_DIR / correlation_training_only/
#   RESULTS_DIR / learning_curves/
#   RESULTS_DIR / train_test_gap/

In [ ]:
!pip install -q catboost i

In [ ]:
"""
Learning curves within the training partition

Run AFTER step 04 (04_model_tuning_and_evaluation): it reuses the training indices and the tuned models
saved there, so the curves use exactly the same 1,960 training samples, the same
16 predictors and the same best hyperparameters. The test partition is never touched.

For each model, sklearn.model_selection.learning_curve refits the tuned configuration
on growing random subsets (10-100%) of each CV training fold and scores it on the
corresponding validation fold (5-fold stratified CV, same folds as the grid search).

Outputs:
  Fig_learning_curves_main.(png|pdf)   CatBoost, GTB, XGBoost, MLP (2 x 2)
  Fig_learning_curves_all.(png|pdf)    all ten models (supplementary, optional)
  learning_curves_long.csv             every point (mean and SD per size)
  learning_curves_summary.csv          one row per model for the text / response letter
"""


import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import learning_curve, StratifiedKFold
from sklearn.preprocessing import LabelEncoder

# ------------------------------------------------------------------ config
DATA_PATH  = DATA_DIR + 'S2_Features_For_Correlation_v3.csv'
SCREEN_DIR = RESULTS_DIR + 'correlation_training_only/'
MODEL_DIR  = RESULTS_DIR + 'train_test_gap/'   # outputs of step 04 (04_model_tuning_and_evaluation)
OUT_DIR    = RESULTS_DIR + 'learning_curves/'
LABEL_COL  = 'class'
SEED       = 42
N_FOLDS    = 5
TRAIN_SIZES = np.linspace(0.1, 1.0, 10)   # fractions of each CV training fold
SCORING    = 'accuracy'

MODELS_MAIN = ['CatBoost', 'GTB', 'XGBoost', 'MLP']
MODELS_ALL  = ['RF', 'GTB', 'CatBoost', 'XGBoost', 'CART', 'NB', 'KNN', 'MDC', 'MLP', 'SVM-RBF']
RUN_ALL     = True            # False = only the four models named by the reviewer
N_JOBS      = {'CatBoost': 1}  # CatBoost already uses all cores internally; others: -1

os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------------ training partition
PREDICTORS = pd.read_csv(SCREEN_DIR + 'v5_final_vif_training.csv')['feature'].tolist()
df = pd.read_csv(DATA_PATH)
X = df[PREDICTORS]
y = pd.Series(LabelEncoder().fit_transform(df[LABEL_COL]), index=df.index)

train_idx = pd.read_csv(MODEL_DIR + 'train_indices.csv')['train_idx'].values
X_train, y_train = X.loc[train_idx], y.loc[train_idx]
assert len(X_train) == 1960, 'Unexpected training size - check MODEL_DIR'

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# ------------------------------------------------------------------ learning curves
models = MODELS_ALL if RUN_ALL else MODELS_MAIN
curves, rows, summary = {}, [], []

for name in models:
    # clone() keeps the tuned hyperparameters and constructor settings (class weights,
    # scaler inside the pipeline) but discards the fitted state.
    est = clone(joblib.load(MODEL_DIR + f'{name}.joblib'))
    print(f'{name}: {len(TRAIN_SIZES) * N_FOLDS} fits ...')
    sizes, tr, va = learning_curve(
        est, X_train, y_train, train_sizes=TRAIN_SIZES, cv=cv, scoring=SCORING,
        shuffle=True, random_state=SEED, n_jobs=N_JOBS.get(name, -1))
    curves[name] = (sizes, tr, va)

    tr_m, tr_s, va_m, va_s = tr.mean(1), tr.std(1), va.mean(1), va.std(1)
    for k, n in enumerate(sizes):
        rows.append({'Model': name, 'n_train': int(n),
                     'train_mean': tr_m[k], 'train_sd': tr_s[k],
                     'cv_mean': va_m[k], 'cv_sd': va_s[k],
                     'gap': tr_m[k] - va_m[k]})
    i80 = int(np.argmin(np.abs(TRAIN_SIZES - 0.8)))
    summary.append({'Model': name,
                    'n_min': int(sizes[0]), 'n_max': int(sizes[-1]),
                    'CV_OA_at_n_min (%)': 100 * va_m[0],
                    'CV_OA_at_n_max (%)': 100 * va_m[-1],
                    'CV_OA_gain_last_20pct (pp)': 100 * (va_m[-1] - va_m[i80]),
                    'Train_OA_at_n_max (%)': 100 * tr_m[-1],
                    'Gap_at_n_max (pp)': 100 * (tr_m[-1] - va_m[-1]),
                    'Gap_at_n_min (pp)': 100 * (tr_m[0] - va_m[0])})
    print(f"   CV OA {100*va_m[0]:.2f}% (n={sizes[0]}) -> {100*va_m[-1]:.2f}% (n={sizes[-1]}) | "
          f"gap {100*(tr_m[0]-va_m[0]):.2f} -> {100*(tr_m[-1]-va_m[-1]):.2f} pp")

pd.DataFrame(rows).to_csv(OUT_DIR + 'learning_curves_long.csv', index=False)
summ = pd.DataFrame(summary).round(2)
summ.to_csv(OUT_DIR + 'learning_curves_summary.csv', index=False)
print('\n', summ.to_string(index=False))

# ------------------------------------------------------------------ figures
STYLE = [('Training', '#0072B2', 'o'), ('Cross-validation', '#D55E00', 's')]  # colour-blind safe

def plot_curves(names, fname, ncols, sharey):
    nrows = int(np.ceil(len(names) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.3 * ncols, 2.8 * nrows),
                             sharex=True, sharey=sharey, squeeze=False)
    for ax, name in zip(axes.flat, names):
        sizes, tr, va = curves[name]
        for scores, (label, color, marker) in zip([tr, va], STYLE):
            m, s = 100 * scores.mean(1), 100 * scores.std(1)
            ax.plot(sizes, m, marker=marker, ms=3.5, lw=1.4, color=color, label=label)
            ax.fill_between(sizes, m - s, m + s, color=color, alpha=0.18, lw=0)
        gap = 100 * (tr.mean(1)[-1] - va.mean(1)[-1])
        ax.set_title(name, fontsize=10, fontweight='bold')
        ax.text(0.97, 0.05, f'Gap at n = {sizes[-1]}: {gap:.1f} pp', transform=ax.transAxes,
                ha='right', va='bottom', fontsize=7.5)
        ax.grid(alpha=0.3, lw=0.5)
        ax.tick_params(labelsize=8)
    for ax in axes.flat[len(names):]:
        ax.set_visible(False)
    for ax in axes[-1]:
        ax.set_xlabel('Training samples', fontsize=9)
    for ax in axes[:, 0]:
        ax.set_ylabel('Overall accuracy (%)', fontsize=9)
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='lower center', ncol=2, frameon=False, fontsize=9)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    fig.savefig(OUT_DIR + fname + '.png', dpi=600, bbox_inches='tight')
    fig.savefig(OUT_DIR + fname + '.pdf', bbox_inches='tight')
    plt.show()

plot_curves(MODELS_MAIN, 'Fig_learning_curves_main', ncols=2, sharey=True)
if RUN_ALL:
    # Own y-axis per panel: NB/MDC sit far below the boosting models.
    plot_curves(MODELS_ALL, 'Fig_learning_curves_all', ncols=5, sharey=False)

print('\nFiles saved to:', OUT_DIR)